# Floci Athena + Glue External Tables Demo with Java

This notebook registers external tables in a Floci-compatible Glue Catalog and queries them with Floci Athena.

It assumes that the first notebook has already written data to these S3-compatible paths:

- `s3://iot-raw/sensehat/temperature_parquet`
- `s3://iot-hudi/sensehat/temperature_hudi`
- `s3://iot-delta/sensehat/temperature_delta`
- `s3://iot-iceberg/warehouse`

The goal is simple:

1. Connect to Floci using the AWS SDK for Java.
2. Create a Glue database.
3. Register external tables.
4. Run `SELECT *` queries with Athena.
5. Print the results.

> Parquet external tables are the safest baseline. Hudi, Delta Lake, and Iceberg support depends on the query engine capabilities exposed by the local Athena-compatible implementation.


## 1. Load Maven Dependencies

In [ ]:
%maven software.amazon.awssdk:s3:2.25.60
%maven software.amazon.awssdk:glue:2.25.60
%maven software.amazon.awssdk:athena:2.25.60


## 2. Configure AWS SDK Clients for Floci

In [ ]:
import java.net.URI;
import java.util.List;

import software.amazon.awssdk.auth.credentials.AwsBasicCredentials;
import software.amazon.awssdk.auth.credentials.StaticCredentialsProvider;
import software.amazon.awssdk.regions.Region;

import software.amazon.awssdk.services.s3.S3Client;
import software.amazon.awssdk.services.s3.model.*;

import software.amazon.awssdk.services.glue.GlueClient;
import software.amazon.awssdk.services.glue.model.*;

import software.amazon.awssdk.services.athena.AthenaClient;
import software.amazon.awssdk.services.athena.model.*;

URI endpoint = URI.create(
    System.getenv().getOrDefault("AWS_ENDPOINT_URL", "http://floci:4566")
);

String accessKey = System.getenv().getOrDefault("AWS_ACCESS_KEY_ID", "test");
String secretKey = System.getenv().getOrDefault("AWS_SECRET_ACCESS_KEY", "test");

var credentials = StaticCredentialsProvider.create(
    AwsBasicCredentials.create(accessKey, secretKey)
);

S3Client s3 = S3Client.builder()
    .endpointOverride(endpoint)
    .region(Region.US_EAST_1)
    .credentialsProvider(credentials)
    .forcePathStyle(true)
    .build();

GlueClient glue = GlueClient.builder()
    .endpointOverride(endpoint)
    .region(Region.US_EAST_1)
    .credentialsProvider(credentials)
    .build();

AthenaClient athena = AthenaClient.builder()
    .endpointOverride(endpoint)
    .region(Region.US_EAST_1)
    .credentialsProvider(credentials)
    .build();

System.out.println("Endpoint: " + endpoint);


## 3. Create Required Buckets

In [ ]:
void createBucketIfMissing(String bucketName) {
    try {
        s3.createBucket(CreateBucketRequest.builder()
            .bucket(bucketName)
            .build());
        System.out.println("Created bucket: " + bucketName);
    } catch (Exception e) {
        System.out.println("Bucket already exists or could not be created: " + bucketName);
    }
}

createBucketIfMissing("iot-raw");
createBucketIfMissing("iot-hudi");
createBucketIfMissing("iot-delta");
createBucketIfMissing("iot-iceberg");
createBucketIfMissing("iot-athena-results");


## 4. Create Glue Database

In [ ]:
String databaseName = "iot";

try {
    glue.createDatabase(CreateDatabaseRequest.builder()
        .databaseInput(DatabaseInput.builder()
            .name(databaseName)
            .description("IoT demo database for Sense HAT Lakehouse tables")
            .build())
        .build());

    System.out.println("Created database: " + databaseName);
} catch (AlreadyExistsException e) {
    System.out.println("Database already exists: " + databaseName);
} catch (Exception e) {
    System.out.println("Could not create database: " + e.getMessage());
}


## 5. Helper Functions

In [ ]:
void deleteTableIfExists(String databaseName, String tableName) {
    try {
        glue.deleteTable(DeleteTableRequest.builder()
            .databaseName(databaseName)
            .name(tableName)
            .build());
        System.out.println("Deleted existing table: " + tableName);
    } catch (EntityNotFoundException e) {
        System.out.println("Table does not exist yet: " + tableName);
    } catch (Exception e) {
        System.out.println("Could not delete table " + tableName + ": " + e.getMessage());
    }
}

List<Column> sensorColumns() {
    return List.of(
        Column.builder().name("deviceId").type("string").build(),
        Column.builder().name("timestamp").type("timestamp").build(),
        Column.builder().name("temperatureCelsius").type("double").build(),
        Column.builder().name("humidityPercent").type("double").build(),
        Column.builder().name("pressureHpa").type("double").build()
    );
}

void createExternalParquetTable(String databaseName, String tableName, String location) {
    deleteTableIfExists(databaseName, tableName);

    StorageDescriptor storageDescriptor = StorageDescriptor.builder()
        .location(location)
        .inputFormat("org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat")
        .outputFormat("org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat")
        .serdeInfo(SerDeInfo.builder()
            .serializationLibrary("org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe")
            .build())
        .columns(sensorColumns())
        .build();

    TableInput tableInput = TableInput.builder()
        .name(tableName)
        .tableType("EXTERNAL_TABLE")
        .parameters(java.util.Map.of(
            "classification", "parquet",
            "EXTERNAL", "TRUE"
        ))
        .storageDescriptor(storageDescriptor)
        .build();

    glue.createTable(CreateTableRequest.builder()
        .databaseName(databaseName)
        .tableInput(tableInput)
        .build());

    System.out.println("Created external Parquet table: " + databaseName + "." + tableName);
}

void createExternalDeltaTable(String databaseName, String tableName, String location) {
    deleteTableIfExists(databaseName, tableName);

    StorageDescriptor storageDescriptor = StorageDescriptor.builder()
        .location(location)
        .columns(sensorColumns())
        .build();

    TableInput tableInput = TableInput.builder()
        .name(tableName)
        .tableType("EXTERNAL_TABLE")
        .parameters(java.util.Map.of(
            "table_type", "DELTA",
            "classification", "delta",
            "EXTERNAL", "TRUE"
        ))
        .storageDescriptor(storageDescriptor)
        .build();

    glue.createTable(CreateTableRequest.builder()
        .databaseName(databaseName)
        .tableInput(tableInput)
        .build());

    System.out.println("Created external Delta table: " + databaseName + "." + tableName);
}

void createExternalHudiTable(String databaseName, String tableName, String location) {
    deleteTableIfExists(databaseName, tableName);

    StorageDescriptor storageDescriptor = StorageDescriptor.builder()
        .location(location)
        .inputFormat("org.apache.hudi.hadoop.HoodieParquetInputFormat")
        .outputFormat("org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat")
        .serdeInfo(SerDeInfo.builder()
            .serializationLibrary("org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe")
            .build())
        .columns(sensorColumns())
        .build();

    TableInput tableInput = TableInput.builder()
        .name(tableName)
        .tableType("EXTERNAL_TABLE")
        .parameters(java.util.Map.of(
            "classification", "hudi",
            "EXTERNAL", "TRUE"
        ))
        .storageDescriptor(storageDescriptor)
        .build();

    glue.createTable(CreateTableRequest.builder()
        .databaseName(databaseName)
        .tableInput(tableInput)
        .build());

    System.out.println("Created external Hudi table: " + databaseName + "." + tableName);
}

void createExternalIcebergTable(String databaseName, String tableName, String location) {
    deleteTableIfExists(databaseName, tableName);

    StorageDescriptor storageDescriptor = StorageDescriptor.builder()
        .location(location)
        .columns(sensorColumns())
        .build();

    TableInput tableInput = TableInput.builder()
        .name(tableName)
        .tableType("EXTERNAL_TABLE")
        .parameters(java.util.Map.of(
            "table_type", "ICEBERG",
            "classification", "iceberg",
            "EXTERNAL", "TRUE"
        ))
        .storageDescriptor(storageDescriptor)
        .build();

    glue.createTable(CreateTableRequest.builder()
        .databaseName(databaseName)
        .tableInput(tableInput)
        .build());

    System.out.println("Created external Iceberg table: " + databaseName + "." + tableName);
}


## 6. Register External Tables

In [ ]:
createExternalParquetTable(
    databaseName,
    "sensehat_parquet",
    "s3://iot-raw/sensehat/temperature_parquet/"
);

createExternalHudiTable(
    databaseName,
    "sensehat_hudi",
    "s3://iot-hudi/sensehat/temperature_hudi/"
);

createExternalDeltaTable(
    databaseName,
    "sensehat_delta",
    "s3://iot-delta/sensehat/temperature_delta/"
);

createExternalIcebergTable(
    databaseName,
    "sensehat_iceberg",
    "s3://iot-iceberg/warehouse/"
);


## 7. List Glue Tables

In [ ]:
var tables = glue.getTables(GetTablesRequest.builder()
    .databaseName(databaseName)
    .build());

tables.tableList().forEach(table -> {
    System.out.println(table.name() + " -> " + table.tableType());
    if (table.storageDescriptor() != null) {
        System.out.println("  location: " + table.storageDescriptor().location());
    }
    System.out.println("  parameters: " + table.parameters());
});


## 8. Athena Query Helpers

In [ ]:
String runAthenaQuery(String sql) throws Exception {
    StartQueryExecutionResponse response = athena.startQueryExecution(
        StartQueryExecutionRequest.builder()
            .queryString(sql)
            .queryExecutionContext(QueryExecutionContext.builder()
                .database(databaseName)
                .build())
            .resultConfiguration(ResultConfiguration.builder()
                .outputLocation("s3://iot-athena-results/")
                .build())
            .build()
    );

    String queryExecutionId = response.queryExecutionId();

    while (true) {
        GetQueryExecutionResponse execution = athena.getQueryExecution(
            GetQueryExecutionRequest.builder()
                .queryExecutionId(queryExecutionId)
                .build()
        );

        QueryExecutionState state = execution.queryExecution().status().state();
        System.out.println("Query " + queryExecutionId + " state: " + state);

        if (state == QueryExecutionState.SUCCEEDED) {
            return queryExecutionId;
        }

        if (state == QueryExecutionState.FAILED || state == QueryExecutionState.CANCELLED) {
            String reason = execution.queryExecution().status().stateChangeReason();
            throw new RuntimeException("Athena query failed: " + reason);
        }

        Thread.sleep(1000);
    }
}

void printAthenaResults(String queryExecutionId) {
    GetQueryResultsResponse results = athena.getQueryResults(
        GetQueryResultsRequest.builder()
            .queryExecutionId(queryExecutionId)
            .build()
    );

    for (Row row : results.resultSet().rows()) {
        System.out.println(
            row.data().stream()
                .map(d -> d.varCharValue() == null ? "NULL" : d.varCharValue())
                .toList()
        );
    }
}

void selectAll(String tableName) {
    System.out.println("\nSELECT * FROM " + tableName);

    try {
        String queryId = runAthenaQuery("""
SELECT *
FROM %s
LIMIT 10
""".formatted(tableName));

        printAthenaResults(queryId);
    } catch (Exception e) {
        System.out.println("Query failed for " + tableName + ": " + e.getMessage());
    }
}


## 9. SELECT * from Each Table

In [ ]:
selectAll("sensehat_parquet");
selectAll("sensehat_hudi");
selectAll("sensehat_delta");
selectAll("sensehat_iceberg");


## 10. Optional: Same Aggregation Query Across Tables

In [ ]:
for (String tableName : List.of("sensehat_parquet", "sensehat_hudi", "sensehat_delta", "sensehat_iceberg")) {
    System.out.println("\nAggregation for table: " + tableName);

    try {
        String queryId = runAthenaQuery("""
SELECT
  deviceId,
  avg(temperatureCelsius) AS avg_temperature,
  avg(humidityPercent) AS avg_humidity,
  avg(pressureHpa) AS avg_pressure
FROM %s
GROUP BY deviceId
""".formatted(tableName));

        printAthenaResults(queryId);
    } catch (Exception e) {
        System.out.println("Aggregation failed for " + tableName + ": " + e.getMessage());
    }
}


## 11. Cleanup

In [ ]:
athena.close();
glue.close();
s3.close();

System.out.println("Done.");


## Notes

Glue acts as the metadata catalog. Athena queries table definitions from Glue and reads the data from S3-compatible storage.

A plain external Parquet table is the safest baseline because it only needs a schema and a location.

Hudi, Delta Lake, and Iceberg are table formats with metadata layers:

- Hudi uses `.hoodie` metadata and commit timelines.
- Delta Lake uses the `_delta_log`.
- Iceberg uses metadata files and snapshots.

A query engine must understand those metadata layers to query the table correctly.
